# QuantGAN Training

This notebook demonstrates how to train a QuantGAN model for financial time series generation.

In [1]:
# --- Notebook setup
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [2]:
!cp -r drive/MyDrive/WGAN/WGAN .
!pip install -e drive/MyDrive/WGAN/WGAN



Obtaining file:///content/drive/MyDrive/WGAN/WGAN
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 358.9/358.9 kB 13.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 9.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 6.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of tensorflow to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.9/297.9 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.5/20.5 MB 74.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from scipy.stats import kurtosis

from quantgan import (
    DataConfig,
    PreprocessConfig,
    DatasetConfig,
    ModelConfig,
    TrainConfig,
    DebugConfig,
    RunConfig,
)
from quantgan.data import DefeatBetaSource, LambertWPreprocessor, DatasetBuilder
from quantgan.evaluation import PaperEvaluator, acf_tf, leverage_tf
from quantgan.training import WGANGPTrainer
from quantgan.utils import set_all_seeds

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
import logging, sys

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    stream=sys.stdout,
    force=True,
)


## 1. Configuration

Set up all training parameters.

In [3]:
# Create configuration objects
# Look at config file for default values

data_cfg = DataConfig(
    ticker="SPY",
    start="2009-05-01",
    end="2018-12-31",
    interval="1d",
)

pre_cfg = PreprocessConfig(
    use_lambert=True,
    renorm_after_lambert=True,
)

ds_cfg = DatasetConfig(
    window_len=127,
    batch_size=30,
    weighted_sampling=True,
    seed=7,
)

model_cfg = ModelConfig(
    generator_type="pure_tcn",  # "pure_tcn" or "svnn"
)

train_cfg = TrainConfig(
    epochs=250,
)

dbg_cfg = DebugConfig()
run_cfg = RunConfig(seed=0)

# Set seeds
set_all_seeds(run_cfg.seed)

## 2. Load and Preprocess Data

In [4]:
# Load data
src = DefeatBetaSource(data_cfg)
df = src.fetch()
close = df["Close"].dropna()
logret = src.log_returns_from_close(df)

print(f"[DATA] {data_cfg.ticker} T={len(logret)} "
      f"mean={float(logret.mean()):.9g} std={float(logret.std()):.9g} "
      f"kurt={float(kurtosis(logret, fisher=False, bias=False)):.3f}")

[DATA] Downloading SPY from DefeatBeta API...


[nltk_data] Downloading package punkt_tab to /tmp/defeatbeta/nltk...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


______      __           _    ______      _        
|  _  \    / _|         | |   | ___ \    | |       
| | | |___| |_ ___  __ _| |_  | |_/ / ___| |_ __ _ 
| | | / _ \  _/ _ \/ _` | __| | ___ \/ _ \ __/ _` |
| |/ /  __/ ||  __/ (_| | |_  | |_/ /  __/ || (_| |
|___/ \___|_| \___|\__,_|\__| \____/ \___|\__\__,_|
📈:: Data Update Time ::	2026-02-20 ::
📈:: Software Version ::	0.0.44      ::
2026-02-22 13:46:29,333 | INFO | DuckDBClient | Cache outdated. Cached: None, Remote: 2026-02-20. Clearing cache...
2026-02-22 13:46:29,371 | INFO | DuckDBClient | httpfs cache cleared
2026-02-22 13:46:29,978 | INFO | DuckDBClient | Cache refreshed. Update time: 2026-02-20
[DATA] Saved to cache: data/defeatbeta_SPY_2009-05-01_2018-12-31_1d.csv
[DATA] SPY T=2433 mean=0.00042953343 std=0.00962302059 kurt=6.975


In [5]:
# Preprocess
pre = LambertWPreprocessor(pre_cfg).fit(logret)
r_train = pre.transform(logret)

print("[PRE] state:", pre.summary())
print(f"[PRE] r_train mean/std = {float(np.mean(r_train)):.4g} {float(np.std(r_train)):.4g}")

[PRE] state: {'use_lambert': True, 'renorm_after_lambert': True, 'r_mean': 0.0004295334295416232, 'r_std': 0.009623020589921769, 'lam_mu': 0.036145429573328836, 'lam_sigma': 0.6655962445571071, 'delta_hat': 0.24627699358959143, 'u_mean': -0.0225337046505423, 'u_std': 0.9997461160580793, 'u_uclip': 2.8060439325483437}
[PRE] r_train mean/std = 3.92e-09 1


## 3. Create Dataset and Targets

In [6]:
# Build dataset
ds_builder = DatasetBuilder(ds_cfg)
train_ds, X_windows, steps_per_epoch = ds_builder.build(r_train)

print(f"[DS] windows {X_windows.shape} | steps_per_epoch: {steps_per_epoch}")

[DS] windows (2307, 127, 1) | steps_per_epoch: 76


In [7]:
# Compute target statistics
real_windows = X_windows.astype(np.float32)
N_TGT = min(int(run_cfg.n_target_windows), int(real_windows.shape[0]))
idx = np.random.RandomState(ds_cfg.seed).choice(
    real_windows.shape[0], size=N_TGT, replace=False
)
real_sub = tf.constant(real_windows[idx], dtype=tf.float32)

acf_abs_real_tgt = tf.reduce_mean(acf_tf(tf.abs(real_sub), run_cfg.vol_lags), axis=0)
acf_sq_real_tgt = tf.reduce_mean(acf_tf(tf.square(real_sub), run_cfg.vol_lags), axis=0)
lev_real_tgt = tf.reduce_mean(leverage_tf(real_sub, run_cfg.lev_lags), axis=0)

print(f"[TGT] shapes: {acf_abs_real_tgt.shape} {acf_sq_real_tgt.shape} {lev_real_tgt.shape}")

[TGT] shapes: (40,) (40,) (40,)


In [8]:
# Create evaluator
evaluator = PaperEvaluator(
    real_series=logret,
    preproc=pre,
    train_cfg=train_cfg,
    dy_base_t=len(logret),
)

print(f"[REAL STATS] mean, std: {evaluator.real_mean:.6g} {evaluator.real_std:.6g}")
print(f"[REAL STATS] quantiles [0.01, 0.05, 0.95, 0.99]: {evaluator.real_qs}")

[REAL STATS] mean, std: 0.000429533 0.00962302
[REAL STATS] quantiles [0.01, 0.05, 0.95, 0.99]: [-0.02816827 -0.01617322  0.01482171  0.02475666]


## 4. Train the Model

In [9]:
# Create trainer
trainer = WGANGPTrainer(
    model_cfg=model_cfg,
    train_cfg=train_cfg,
    debug_cfg=dbg_cfg,
    window_len=ds_cfg.window_len,
    steps_per_epoch=steps_per_epoch,
    vol_lags=run_cfg.vol_lags,
    lev_lags=run_cfg.lev_lags,
    acf_abs_real_tgt=acf_abs_real_tgt,
    acf_sq_real_tgt=acf_sq_real_tgt,
    lev_real_tgt=lev_real_tgt,
)

print("Trainer created successfully!")

Trainer created successfully!


In [10]:
# Train
out_dir = os.path.join(os.getcwd(), run_cfg.out_dir)

run_out = trainer.train(
    train_ds=train_ds,
    evaluator=evaluator,
    real_logret=logret,
    out_dir=out_dir,
    run_seed=run_cfg.seed,
    gen_type=model_cfg.generator_type,
    n_plot_runs=run_cfg.n_plot_runs,
)


print("\n=== TRAINING COMPLETE ===")
print(f"Best epoch: {run_out['best_epoch']}")
print(f"Best score: {run_out['best_score']:.6g}")
print(f"Weights saved to: {run_out['best_weights_path']}")

2026-02-22 13:47:21,469 | INFO | quantgan.training.trainer | [TRAIN] Starting training for 250 epochs
2026-02-22 13:47:21,473 | INFO | quantgan.training.trainer | [MODEL] Generator: 210161 params, Discriminator: 58 params
2026-02-22 13:47:40,845 | INFO | quantgan.training.trainer | [PRETRAIN D] Epoch 00 | loss_D=76.11 W=-0.2998 GP=7.641 | gp_norm_mean=3.749 gp_norm_max=4.477
2026-02-22 13:48:07,385 | INFO | quantgan.training.trainer | [Epoch 00] → Saved BEST generator (paper_score=0.895208)
2026-02-22 13:48:07,386 | INFO | quantgan.training.trainer | [Epoch 00] Complete | loss_D=0 loss_G=0 | PAPER_score=0.895208 acf_abs=0.0695 acf_x=0.0278 acf_sq=0.06 lev=0.475 dy_sum=2.76e+03 DY1=116 DY5=465 DY20=2.18e+03 DY100=1.92 | raw_mean=0.0129 raw_std=0.0128 mean_err=1.3 std_err=0.33 q_err=2.27
2026-02-22 13:48:07,815 | INFO | quantgan.training.trainer | [PRETRAIN D] Epoch 01 | loss_D=-12.26 W=-17.08 GP=0.4819 | gp_norm_mean=1.680 gp_norm_max=1.881
2026-02-22 13:48:25,870 | INFO | quantgan.trai